# examples-seen-step-axis — worked example 3: wandb Log Payload Must Contain examples_seen Key

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `examples-seen-step-axis`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When logging to Weights & Biases, the x-axis must be explicitly included in every `wandb.log(...)` payload as a key. Simply computing `examples_seen` locally without including it in the payload means it never reaches the tracker. The standard pattern is `wandb.log({'train_loss': loss, 'examples_seen': step * batch_size})` — both metrics in the same call.

## Worked solution

We build a mock wandb object that records every payload, then simulate a training loop and verify that every logged payload contains the `examples_seen` key.

**Step 1 — mock wandb:** A simple class that stores all payloads passed to `.log()` in a list.

**Step 2 — training loop:** For each step, compute `examples_seen = step * batch_size` and call `wandb.log({'loss': loss, 'examples_seen': examples_seen})`.

**Step 3 — verification:** Check that every logged payload has both `'loss'` and `'examples_seen'` keys, and that the `examples_seen` values are exactly `[batch_size, 2*batch_size, 3*batch_size, ...]`.

This is the pattern the ARENA tests check: not just that the values are right, but that the key is present in every single log call.

In [ ]:
import torch as t

t.manual_seed(22)

class MockWandb:
    def __init__(self):
        self.calls = []
    def log(self, payload):
        self.calls.append(dict(payload))

def train_loop_with_logging(losses, batch_size, wandb_obj):
    for step, loss in enumerate(losses, start=1):
        examples_seen = step * batch_size
        wandb_obj.log({'train_loss': loss, 'examples_seen': examples_seen})
    return len(losses)

# Simulate 5-step training with bs=64
batch_size = 64
losses = [2.0 - 0.3 * i for i in range(5)]
wandb = MockWandb()
n = train_loop_with_logging(losses, batch_size, wandb)

print(f"Logged {n} steps")
for i, payload in enumerate(wandb.calls):
    assert 'train_loss' in payload, f"Step {i+1}: missing 'train_loss'"
    assert 'examples_seen' in payload, f"Step {i+1}: missing 'examples_seen'"
    expected_ex = (i + 1) * batch_size
    assert payload['examples_seen'] == expected_ex, (
        f"Step {i+1}: expected examples_seen={expected_ex}, got {payload['examples_seen']}"
    )
    print(f"  Step {i+1}: loss={payload['train_loss']:.2f}, examples_seen={payload['examples_seen']}")

print("All payloads contain the examples_seen key with correct values.")